###  Image processing

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install --upgrade --no-deps tensorflow==2.17.0 opencv-python==4.9.0.80 numpy==1.26.4 pydicom nibabel dicompyler-core matplotlib scikit-learn seaborn tqdm

In [ ]:
!pip install -q pydicom==2.3.1 rt-utils SimpleITK


In [ ]:
import numpy as np
from collections import defaultdict
import pandas as pd
import tensorflow as tf
import cv2
import os
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import nibabel as nib
from ipywidgets import interact, IntSlider
from tqdm import tqdm


print("Pydicom:", pydicom.__version__)
print("SimpleITK:", sitk.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

**Organizing image folders**

```
NSCLC/NSCLC-Radiomics/
    LUNG1-001/
        09-18-2008-StudyID-NA-69331/
            0.000000-NA-82046/ <-- Reconstructed CT with lung filter (high resolution, thin slices)
            3.000000-NA-78236/ <-- Contains patient contours or ROI, usually drawn in radiotherapy software. It is not a series of images, but structures superimposed on the CT.
            300.000000-Segmentation-9.554/ <-- Segmentation mask (SEG) - Defines the tumor region
```

Each patient has a LUNG1-XXX file.

Within it, each study has a folder with the study date.

DICOM: Image data

Computed Tomography (CT): A set of axial slices representing the patient's anatomy. Each CT slice is 3 mm thick.

Segmentation (SEG): Masks that define regions of interest.
Radiotherapy Structures (RTSTRUCT): Outlines drawn by physicians for radiotherapy planning. Associated with a specific CT series.

In [ ]:
# Check the image data types

base_dir = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-027/01-01-2014-StudyID-NA-35913"

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        first_file = os.listdir(folder_path)[0]
        dcm_path = os.path.join(folder_path, first_file)
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        print(folder, "→", ds.Modality)
        print("SeriesDescription:", ds.get("SeriesDescription", "N/A"))
        print("ProtocolName:", ds.get("ProtocolName", "N/A"))
        print("SeriesNumber:", ds.get("SeriesNumber", "N/A"))
        print("Contrast/Bolus Agent:", ds.get("ContrastBolusAgent", "N/A"))
        print("Contrast/Bolus Administration Route:", ds.get("ContrastBolusRoute", "N/A"))


Output - Check the image data types

```
300.000000-Segmentation-8.487 → SEG
SeriesDescription: Segmentation
ProtocolName: N/A
SeriesNumber: 300
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
2.000000-NA-63878 → RTSTRUCT
SeriesDescription: N/A
ProtocolName: N/A
SeriesNumber: 2
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
1.000000-NA-45865 → CT
SeriesDescription: N/A
ProtocolName: N/A
SeriesNumber: 1
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
```

#### 1. Format conversion

In [ ]:

# Libraries: os, pydicom, SimpleITK, collections


# Standardized DICOM UIDs
CT_UID = "1.2.840.10008.5.1.4.1.1.2"
SEG_UID = "1.2.840.10008.5.1.4.1.1.66.4"


def find_series(patient_path):
    ct_series = defaultdict(list)
    seg_file = None
    ref_uid_seg = None

    for root, _, files in os.walk(patient_path):
        for f in files:
            if not f.lower().endswith(".dcm"):
                continue

            dcm_path = os.path.join(root, f)
            try:
                # Read only the header
                d = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                sop = d.SOPClassUID

                if sop == CT_UID:
                    series_uid = d.SeriesInstanceUID
                    ct_series[series_uid].append(dcm_path)

                elif sop == SEG_UID:
                    seg_file = dcm_path
                    # Attempts to retrieve the UID referenced by SEG
                    if hasattr(d, 'ReferencedSeriesSequence') and d.ReferencedSeriesSequence:
                        ref_uid_seg = d.ReferencedSeriesSequence[0].SeriesInstanceUID

            except Exception:
                pass

    if not ct_series:
        return None, None

    
    if seg_file and ref_uid_seg and ref_uid_seg in ct_series:
        chosen_series_uid = ref_uid_seg
    # Select the series with the most slices
    elif ct_series:
        chosen_series_uid = max(ct_series, key=lambda k: len(ct_series[k]))
    else:
        return None, None

    chosen_ct_folder = os.path.dirname(ct_series[chosen_series_uid][0])

    return chosen_ct_folder, seg_file

# Load CT as a volume
def load_ct(ct_folder):
    reader = sitk.ImageSeriesReader()
    dicom_files = reader.GetGDCMSeriesFileNames(ct_folder)
    reader.SetFileNames(dicom_files)
    return reader.Execute()

#  Load and align the SEG mask
def load_seg(seg_file, ct_img):
    try:
        # Carrega o volume SEG
        seg = sitk.ReadImage(seg_file)
        seg_resampled = sitk.Resample(
            seg,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0
        )

        return sitk.Cast(seg_resampled, sitk.sitkUInt8)

    except Exception as e:
        print(f"Error in SEG's final alignment: {e}. Returning an empty mask.")
        empty_mask = sitk.Image(ct_img.GetSize(), sitk.sitkUInt8)
        empty_mask.CopyInformation(ct_img)
        return empty_mask

# Save as NIfTI (.nii.gz)
def save_nii(image, path):
    sitk.WriteImage(image, path)
    print(f"Save: {path}")


root = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics"
output_folder_name = "nii_aligned"

for patient in sorted(os.listdir(root)):
    patient_path = os.path.join(root, patient)
    if not os.path.isdir(patient_path) or not patient.startswith("LUNG1-"):
        continue

    print(f"\n Patient {patient}")

    output_path = os.path.join(patient_path, output_folder_name)
    os.makedirs(output_path, exist_ok=True)

    ct_folder, seg_file = find_series(patient_path)

    if ct_folder is None:
        print("No CT series found.")
        continue

    if seg_file is None:
        print("No SEG files found.")
        continue

    print(f"Selected CT: {ct_folder.split('/')[-1]}")

    try:
        
        ct_volume = load_ct(ct_folder)

        seg_volume = load_seg(seg_file, ct_volume)

        save_nii(ct_volume, os.path.join(output_path, "ct_aligned.nii.gz"))
        save_nii(seg_volume, os.path.join(output_path, "seg_aligned.nii.gz"))

    except Exception as e:
        print(f"Error while processing {patient}: {e}")

**Verification of the anatomical orientation of the images**

Negative X-axis → Right (R)

Negative Y-axis → Anterior (A)

Positive Z-axis → Superior (S)

It was verified that the images are already in the correct orientation.

In [ ]:
def print_meta(name, img):
    print(name)
    print(" Size:", img.GetSize())
    print(" Spacing:", img.GetSpacing())
    print(" Origin:", img.GetOrigin())
    print(" Direction:", img.GetDirection())
    print(" PixelID:", img.GetPixelIDTypeAsString())
    print("")

ct = sitk.ReadImage(ct_path)
seg = sitk.ReadImage(seg_path)

print_meta("CT:", ct)
print_meta("SEG:", seg)


#### 2. Resample

1 x 1 x 1 mm Resampling

Objective: To standardize the voxel size across different CT scans. This allows for high-quality image reconstruction in multiple planes (axial, coronal, and sagittal) with minimal distortion or loss of detail.

In [ ]:
"""
Libraries: SimpleITK, numpy, os
Objective: Resample CT and segment to 1 mm × 1 mm × 1 mm (isotropic)
"""

# Function for 1x1x1 isotropic resampling
def resample_image(itk_image, new_spacing=[1.0, 1.0, 1.0], is_label=False):

    original_spacing = itk_image.GetSpacing()
    original_size = itk_image.GetSize()

    new_size = [
        int(round(original_size[i] * (original_spacing[i] / new_spacing[i])))
        for i in range(3)
    ]

    interpolator = sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(interpolator)
    resampler.SetOutputDirection(itk_image.GetDirection())
    resampler.SetOutputOrigin(itk_image.GetOrigin())
    resampler.SetDefaultPixelValue(0)

    return resampler.Execute(itk_image)


BASE_DIR = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/"
patients = sorted([p for p in os.listdir(BASE_DIR) if p.startswith("LUNG1-")])

print("Total number of patients found:", len(patients))

for patient in patients:

    print(f"\n=== Processing {patient} ===")

    ct_path  = os.path.join(BASE_DIR, patient, "nii_aligned", "ct_aligned.nii.gz")
    seg_path = os.path.join(BASE_DIR, patient, "nii_aligned", "seg_aligned.nii.gz")

    if not (os.path.exists(ct_path) and os.path.exists(seg_path)):
        print("CT or Segmentation not found.")
        continue

    ct  = sitk.ReadImage(ct_path)
    seg = sitk.ReadImage(seg_path)

    # Resample CT and SEG
    ct_resampled  = resample_image(ct,  new_spacing=[1.0, 1.0, 1.0], is_label=False)
    seg_resampled = resample_image(seg, new_spacing=[1.0, 1.0, 1.0], is_label=True)

    out_dir = os.path.join(BASE_DIR, patient, "resampled")
    os.makedirs(out_dir, exist_ok=True)

    sitk.WriteImage(ct_resampled,  os.path.join(out_dir, "ct_resampled.nii.gz"))
    sitk.WriteImage(seg_resampled, os.path.join(out_dir, "seg_resampled.nii.gz"))
    print(f"1x1x1 resample saved to: {out_dir}")


**Viewing the images after resampling**

```

patient_dir = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-001/resampled"

def load_resampled(path):
    ct_path_npy  = os.path.join(path, "ct_resampled.npy")
    seg_path_npy = os.path.join(path, "seg_resampled.npy")

    ct_path_nii  = os.path.join(path, "ct_resampled.nii.gz")
    seg_path_nii = os.path.join(path, "seg_resampled.nii.gz")

    if os.path.exists(ct_path_npy):
        ct  = np.load(ct_path_npy)
        seg = np.load(seg_path_npy)
        return ct, seg

    elif os.path.exists(ct_path_nii):
        ct_img  = sitk.ReadImage(ct_path_nii)
        seg_img = sitk.ReadImage(seg_path_nii)
        ct  = sitk.GetArrayFromImage(ct_img)
        seg = sitk.GetArrayFromImage(seg_img)
        return ct, seg

    else:
        raise FileNotFoundError("No files found.")

ct, seg = load_resampled(patient_dir)

print("CT:", ct.shape)
print("SEG:", seg.shape)


mid = 239

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(ct[mid], cmap="gray")
plt.axis("off")

plt.figure(figsize=(6,6))
plt.imshow(ct[mid], cmap="gray")
plt.imshow(seg[mid], alpha=0.4)
plt.axis("off")

plt.tight_layout()
plt.savefig("ct_segp001_2.png", bbox_inches="tight", pad_inches=0)
plt.savefig("ct_segp001_2.png")
plt.show()

```

Patient 01

<img src="/img/Resample_LUNG1-001.png" width="350">



In [ ]:
# Interactive slice viewer

@interact(z=IntSlider(min=0, max=ct.shape[0]-1, value=mid, step=1))
def view_slice(z):
    plt.figure(figsize=(6,6))
    plt.imshow(ct[z], cmap="gray")
    plt.imshow(seg[z], alpha=0.4)
    plt.title(f"Slice {z}")
    plt.axis("off")
    plt.show()


def save_png_slices(ct, seg, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(ct.shape[0]):
        plt.figure(figsize=(6,6))
        plt.imshow(ct[i], cmap="gray")
        plt.imshow(seg[i], alpha=0.4)
        plt.axis("off")
        plt.savefig(os.path.join(out_dir, f"slice_{i:03d}.png"),
                    bbox_inches="tight", pad_inches=0)
        plt.close()

# save_png_slices(ct, seg, os.path.join(patient_dir, "png_slices"))

In [ ]:
"""
Check if the resampling was successful
Libraries: nibabel, numpy
"""

ct_path = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-018/resampled/ct_resampled.nii.gz"
seg_path = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-018/resampled/seg_resampled.nii.gz"


ct_img = nib.load(ct_path)
seg_img = nib.load(seg_path)
ct_np = ct_img.get_fdata()
seg_np = seg_img.get_fdata()

# Dimensions
print(ct_np.shape)
print(seg_np.shape)

# Voxel spacing 
print("Spacing CT:", ct_img.header["pixdim"][1:4])
print("Spacing SEG:", seg_img.header["pixdim"][1:4])

# Affine
print("\nAffine CT:\n", ct_img.affine)
print("\nAffine SEG:\n", seg_img.affine)

Output - Check if the resampling was successful

```
(500, 500, 342)
(500, 500, 342)
Spacing CT: [1. 1. 1.]
Spacing SEG: [1. 1. 1.]

Affine CT:
 [[  -1.            0.            0.          250.11199951]
 [   0.           -1.            0.          250.11199951]
 [   0.            0.            1.         -187.8999939 ]
 [   0.            0.            0.            1.        ]]

Affine SEG:
 [[  -1.            0.            0.          250.11199951]
 [   0.           -1.            0.          250.11199951]
 [   0.            0.            1.         -187.8999939 ]
 [   0.            0.            0.            1.        ]]
```

#### 3. Global Clipping, Multiple Windowing, Normalization of Intensity, Bounding Box, Resizing

In [ ]:
"""
Libraries: os, numpy, nibabel, tqdm
Objective: Remove outliers, create two volumes (soft tissue and lung parenchyma), normalize, combine the volumes
into a single array, locate the tumor region (Bounding Box), reduce the volume to include only
the tumor, standardize the size of all images to 128 x 128 x 128, binarize the mask, and store the
in npy format

"""

def clip_hu(ct, min_hu=-1024, max_hu=1024):
    return np.clip(ct, min_hu, max_hu)

def window_hu(ct, low, high):
    return np.clip(ct, low, high)

def get_bounding_box(mask, margin=5):
    coords = np.array(np.where(mask > 0))
    if coords.size == 0:
        return None

    min_coords = coords.min(axis=1) - margin
    max_coords = coords.max(axis=1) + margin

    min_coords = np.maximum(min_coords, 0)
    max_coords = np.minimum(max_coords, np.array(mask.shape) - 1)

    return tuple(min_coords), tuple(max_coords + 1)

def crop_to_bbox(img, min_coords, max_coords):
    z1, y1, x1 = min_coords
    z2, y2, x2 = max_coords
    return img[z1:z2, y1:y2, x1:x2]

def pad_or_crop_to_shape(arr, target_shape):

    current_shape = np.array(arr.shape)
    target_shape = np.array(target_shape)
    is_4d = (arr.ndim == 4)

    if is_4d:
        result = np.zeros(tuple(target_shape) + (arr.shape[-1],), dtype=arr.dtype)
    else:
        result = np.zeros(tuple(target_shape), dtype=arr.dtype)

    crop_slices = []
    pad_slices = []

    for i in range(3):
        size = current_shape[i]
        target = target_shape[i]

        if size >= target:
            start = (size - target) // 2
            crop_slices.append(slice(start, start + target))
            pad_slices.append(slice(0, target))
        else:
            crop_slices.append(slice(0, size))
            start = (target - size) // 2
            pad_slices.append(slice(start, start + size))

    if is_4d:
        result[pad_slices[0], pad_slices[1], pad_slices[2], :] = \
            arr[crop_slices[0], crop_slices[1], crop_slices[2], :]
    else:
        result[pad_slices[0], pad_slices[1], pad_slices[2]] = \
            arr[crop_slices[0], crop_slices[1], crop_slices[2]]

    return result

def normalize_minmax(ct, low, high):
    ct = ct.astype(np.float32)
    denom = high - low
    if denom == 0:
        return np.zeros_like(ct)
    return (ct - low) / denom


def process_patient_final(ct_path, seg_path, out_ct, out_seg, target_shape=(128,128,128)):

    # Parameters
    MIN_GLOBAL, MAX_GLOBAL = -1200, 600
    W_MIN_SOFT, W_MAX_SOFT = -100, 300
    W_MIN_LUNG, W_MAX_LUNG = -1000, 200


    ct = nib.load(ct_path).get_fdata()
    seg = nib.load(seg_path).get_fdata()

    # Clipping Global
    ct = clip_hu(ct, MIN_GLOBAL, MAX_GLOBAL)

    # Janelamento (Soft + Lung)
    ct_soft = window_hu(ct, W_MIN_SOFT, W_MAX_SOFT)
    ct_lung = window_hu(ct, W_MIN_LUNG, W_MAX_LUNG)

    # Normalization
    ct_soft = normalize_minmax(ct_soft, W_MIN_SOFT, W_MAX_SOFT)
    ct_lung = normalize_minmax(ct_lung, W_MIN_LUNG, W_MAX_LUNG)


    ct_multi = np.stack([ct_soft, ct_lung], axis=-1)

    # Bounding Box
    bbox = get_bounding_box(seg)
    if bbox is None:
        print(f"⚠ Sem tumor em {ct_path} — ignorado.")
        return
    minc, maxc = bbox

    ct_crop = crop_to_bbox(ct_multi, minc, maxc)
    seg_crop = crop_to_bbox(seg, minc, maxc)

    # Pad/Crop for fixed shape
    ct_final = pad_or_crop_to_shape(ct_crop, target_shape)
    seg_final = pad_or_crop_to_shape(seg_crop, target_shape)

    # Binarize
    seg_final = (seg_final > 0).astype(np.uint8)

    # Save
    np.save(out_ct, ct_final)   
    np.save(out_seg, seg_final)

    print(f"Processed: {out_ct}")


base_dir = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/"
patients = sorted([p for p in os.listdir(base_dir) if p.startswith("LUNG1-")])

target_shape = (128,128,128)

for p in tqdm(patients):
    resampled_dir = os.path.join(base_dir, p, "resampled")
    out_dir = os.path.join(base_dir, p, "processed_final")
    os.makedirs(out_dir, exist_ok=True)

    ct_path = os.path.join(resampled_dir, "ct_resampled.nii.gz")
    seg_path = os.path.join(resampled_dir, "seg_resampled.nii.gz")

    if not os.path.exists(ct_path) or not os.path.exists(seg_path):
        continue

    out_ct = os.path.join(out_dir, "ct_multi.npy")
    out_seg = os.path.join(out_dir, "seg.npy")

    process_patient_final(ct_path, seg_path, out_ct, out_seg, target_shape)


